# Base de Datos 2025 - EES N°2 "Escuela de Educación Secundaria N°2 Fermín González Ancalao"
### Por Amilcar Jesús Fernández

El proyecto surge como base de entrenamiento, y a su vez, con la necesidad de transformar un paquete aproximado de 250 archivos de extenciones variadas, en su mayoría .doc/docx, en una base de datos robusta, consistente y escalable en SQL.
Los archivos pertencen al ciclo lectivo 2025, aunque se observaron archivos de años anteriores. Cada archivo de word contiene normalmente una tabla, aunque existen archivos con varias tablas desconectadas, relacionadas entre sí sólo por contexto.

In [1]:
import pandas as pd
from docx import Document
import re
import sqlite3

# ETAPA 1 — EXTRACCIÓN RAW (Fuente: Datos)

## Objetivo
Construir `df_datos_raw` preservando el contenido del archivo Word tal como viene en origen, sin aplicar todavía limpieza ni transformación de negocio.

## Archivo piloto
`1°1° 2025 Datos.docx`

## Entrada
- Documento Word `.docx`
- 1 tabla detectada
- 12 columnas identificadas en estructura fuente

## Proceso de esta etapa
1. Abrir documento.
2. Detectar tabla.
3. Recorrer filas (excluyendo encabezado).
4. Extraer valores crudos por celda.
5. Agregar metadatos de contexto:
   - curso
   - division
   - anio
   - fuente
   - archivo_origen

## Regla metodológica
En esta etapa:

✔ preservar datos originales  
✔ capturar campos `_raw`  
✔ no limpiar  
✔ no separar campos compuestos  
✔ no validar todavía

No se hace aún:
- STG
- validación
- reconciliación

## Output esperado
DataFrame:

`df_datos_raw`

Columnas esperadas:

- nombre_original  
- libro_matriz_raw  
- folio_raw  
- legajo_raw  
- escuela_procedencia_raw  
- dni_raw  
- fecha_nacimiento_raw  
- lugar_nacimiento_raw  
- edad_raw  
- direccion_raw  
- telefono_raw  
- curso  
- division  
- anio  
- fuente  
- archivo_origen

In [2]:
ruta_datos = r"C:\Users\amilc\OneDrive\Proyecto Data Science Media 2\EES Nº 2 2025-20260117T072850Z-3-001\EES Nº 2 2025\Datos 2025\Datos\1°1° 2025 Datos.docx"

doc_datos = Document(ruta_datos)

In [3]:
len(doc_datos.tables)

1

In [4]:
tabla_datos = doc_datos.tables[0]

In [5]:
len(tabla_datos.rows),
len(tabla_datos.rows[0].cells)

12

In [6]:
registros_datos_raw = []

In [7]:
for fila in tabla_datos.rows[1:]:
    
    registro = {
        "nombre_original": fila.cells[1].text,
        "libro_matriz_raw": fila.cells[2].text,
        "folio_raw": fila.cells[3].text,
        "legajo_raw": fila.cells[4].text,
        "escuela_procedencia_raw": fila.cells[5].text,
        "dni_raw": fila.cells[6].text,
        "fecha_nacimiento_raw": fila.cells[7].text,
        "lugar_nacimiento_raw": fila.cells[8].text,
        "edad_raw": fila.cells[9].text,
        "direccion_raw": fila.cells[10].text,
        "telefono_raw": fila.cells[11].text,
        "curso": "1",
        "division": "1",
        "anio": "2025",
        "fuente": "Datos",
        "archivo_origen": "1°1° 2025 Datos.docx"
    }

    registros_datos_raw.append(registro)

In [8]:
len(registros_datos_raw)

27

In [9]:
registros_datos_raw[0]

{'nombre_original': ' ACHIGAR, Tiziano',
 'libro_matriz_raw': '42',
 'folio_raw': '240',
 'legajo_raw': '31/25',
 'escuela_procedencia_raw': '25',
 'dni_raw': '52746170',
 'fecha_nacimiento_raw': '26/12/12',
 'lugar_nacimiento_raw': 'Punta Alta',
 'edad_raw': '12',
 'direccion_raw': 'Humberto 2538',
 'telefono_raw': '2932-520752M',
 'curso': '1',
 'division': '1',
 'anio': '2025',
 'fuente': 'Datos',
 'archivo_origen': '1°1° 2025 Datos.docx'}

In [10]:
df_datos_raw = pd.DataFrame(registros_datos_raw)

In [11]:
df_datos_raw.head()

,nombre_original,libro_matriz_raw,folio_raw,legajo_raw,escuela_procedencia_raw,dni_raw,fecha_nacimiento_raw,lugar_nacimiento_raw,edad_raw,direccion_raw,telefono_raw,curso,division,anio,fuente,archivo_origen
0,"ACHIGAR, Tiziano",42,240,31/25,25,52746170,26/12/12,Punta Alta,12,Humberto 2538,2932-520752M,1,1,2025,Datos,1°1° 2025 Datos.docx
1,"AGUILAR, Ian Nicolás",42,241,32/25,4,52778569,27/11/12,Punta Alta,12,Bº Albatros XXVII c393,2932-573441M,1,1,2025,Datos,1°1° 2025 Datos.docx
2,"ALTAMIRANO, Joaquin Román",42,242,33/25,8,52887715,14/01/13,San Juan,12,Humberto 1916,2932-614538M,1,1,2025,Datos,1°1° 2025 Datos.docx
3,"ARANDA, Maia Mariel",42,243,34/25,1,52617713,06/07/12,Punta Alta,12,Entre Ríos 1899,2932-520048P,1,1,2025,Datos,1°1° 2025 Datos.docx
4,"CARRERAS FLAMMINI, Paula Sofía",42,244,35/25,8,52961006,01/02/13,Juris Fed,12,Humberto 2447,2932-505154M,1,1,2025,Datos,1°1° 2025 Datos.docx


In [12]:
df_datos_raw.shape

(27, 16)

# ETAPA 2 — STAGING (Normalización y Transformación Estructural)

## Objetivo
Construir `df_datos_stg` a partir de `df_datos_raw`,
preservando los campos raw y creando columnas normalizadas listas para validación.

## Entrada
DataFrame origen:

`df_datos_raw`

## Alcance de esta etapa

### Normalización técnica
Aplicar:

- trim / strip bordes
- colapso de espacios múltiples
- tratamiento consistente de nulos
- conservación de campos `_raw`

Ejemplo:

```text
"  ACHIGAR,   Tiziano " → "ACHIGAR, Tiziano"
```

---

### Transformación estructural
Separar campos compuestos.

Ejemplo:

```text
telefono_raw
"2914551234 M"
```

se transforma en:

```text
telefono_num = "2914551234"
tipo_contacto = "M"
```

---

## Output esperado

DataFrame:

`df_datos_stg`

Columnas principales:

- nombre_original
- nombre_completo

- libro_matriz_raw
- libro_matriz

- folio_raw
- folio

- legajo_raw
- legajo

- telefono_raw
- telefono_num
- tipo_contacto

+ contexto (se incorpora de la etapa anterior sin modificaciones):
- curso
- division
- anio
- fuente
- archivo_origen

---

## Regla metodológica
En esta etapa:

✔ normalizar  
✔ transformar  
✔ preservar raw  
✔ no validar todavía

No se hace aún:
- reconciliación
- matching
- exportación SQL

In [13]:
df_datos_stg = df_datos_raw.copy()

### Normalización de los nombres de alumnos

In [14]:
df_datos_stg["nombre_completo"] = df_datos_stg["nombre_original"].apply(
    lambda x: re.sub(r"\s+", " ",x.strip())
)

In [15]:
df_datos_stg[["nombre_original","nombre_completo"]].head()

,nombre_original,nombre_completo
0,"ACHIGAR, Tiziano","ACHIGAR, Tiziano"
1,"AGUILAR, Ian Nicolás","AGUILAR, Ian Nicolás"
2,"ALTAMIRANO, Joaquin Román","ALTAMIRANO, Joaquin Román"
3,"ARANDA, Maia Mariel","ARANDA, Maia Mariel"
4,"CARRERAS FLAMMINI, Paula Sofía","CARRERAS FLAMMINI, Paula Sofía"


### Normalización de identificadores administrativos

Se construyen versiones normalizadas de:

- libro_matriz
- folio
- legajo

Preservando sus campos `_raw` originales.

Regla:
Estos campos se tratan como identificadores administrativos (texto),
no como variables numéricas para cálculo.

In [16]:
df_datos_stg["libro_matriz"] = df_datos_stg["libro_matriz_raw"].apply(
    lambda x: re.sub(r"\s+", " ",x.strip())
)

df_datos_stg["folio"] = df_datos_stg["folio_raw"].apply(
    lambda x: re.sub(r"\s+", " ",x.strip())
)

df_datos_stg["legajo"] = df_datos_stg["legajo_raw"].apply(
    lambda x: re.sub(r"\s+", " ",x.strip())
)

In [17]:
df_datos_stg[["libro_matriz_raw","libro_matriz","folio_raw","folio","legajo_raw","legajo"]].head()

,libro_matriz_raw,libro_matriz,folio_raw,folio,legajo_raw,legajo
0,42,42,240,240,31/25,31/25
1,42,42,241,241,32/25,32/25
2,42,42,242,242,33/25,33/25
3,42,42,243,243,34/25,34/25
4,42,42,244,244,35/25,35/25


### Transformación estructural de teléfonos

Separación de campo compuesto:

`telefono_raw`

en dos columnas derivadas:

- telefono_num
- tipo_contacto

Convención:
- M = Madre
- P = Padre

Se preserva siempre:
`telefono_raw`

In [18]:
def separar_telefono(valor):
    valor = str(valor).strip()

    if valor[-1].isalpha():
        tipo_contacto = valor[-1].upper()
        telefono_base = valor[:-1]
    else:
        tipo_contacto = None
        telefono_base = valor

    telefono_num = re.sub(r"\D", "", telefono_base)

    return pd.Series({
        "telefono_num": telefono_num,
        "tipo_contacto": tipo_contacto
    })

In [19]:
df_datos_stg[["telefono_num","tipo_contacto"]] = (df_datos_stg["telefono_raw"].apply(separar_telefono))

In [20]:
df_datos_stg[["telefono_raw","telefono_num","tipo_contacto"]].head(27)

,telefono_raw,telefono_num,tipo_contacto
0,2932-520752M,2932520752,M
1,2932-573441M,2932573441,M
2,2932-614538M,2932614538,M
3,2932-520048P,2932520048,P
4,2932-505154M,2932505154,M
5,2932-410841M,2932410841,M
6,2932474863/441518M,2932474863441518,M
7,2932-404188M,2932404188,M
8,2932-447160/441779M,2932447160441779,M
9,2932-635639M,2932635639,M


### Profiling adicional de anomalías telefónicas

Se detectan dos tipos de casos:

1. multivaluados explícitos (`/`)
2. posible concatenación anómala de múltiples teléfonos
   por longitud atípica del número.

#### Detección de campos multivaluados en teléfonos

Se identifican registros con múltiples teléfonos en un mismo campo
(separados por "/").

No se separan todavía.

Se marcan para revisión/control y futura normalización en entidad contactos.

In [21]:
df_datos_stg["telefono_multiple_flag"] = (df_datos_stg["telefono_raw"].str.contains("/"))

In [22]:
df_datos_stg[df_datos_stg["telefono_multiple_flag"]][["telefono_raw","telefono_num","tipo_contacto"]]

,telefono_raw,telefono_num,tipo_contacto
6,2932474863/441518M,2932474863441518,M
8,2932-447160/441779M,2932447160441779,M


In [23]:
df_datos_stg["telefono_largo"] = (
    df_datos_stg["telefono_num"].str.len()
)

In [24]:
df_datos_stg["telefono_largo"].value_counts().sort_index()

telefono_largo
10    23
11     1
16     2
17     1
Name: count, dtype: int64

In [25]:
df_datos_stg["telefono_anomalo_flag"] = (
    df_datos_stg["telefono_largo"] > 11
)

In [26]:
df_datos_stg[df_datos_stg["telefono_anomalo_flag"]][["telefono_raw","telefono_num","telefono_largo"]]

,telefono_raw,telefono_num,telefono_largo
6,2932474863/441518M,2932474863441518,16
8,2932-447160/441779M,2932447160441779,16
16,2932-5026617416120P,29325026617416120,17


### Normalización de campos adicionales

Se aplican reglas básicas de limpieza a:

- escuela_procedencia
- dni
- fecha_nacimiento
- lugar_nacimiento
- edad
- direccion

Objetivo:
homogeneizar formato sin alterar significado.

In [27]:
cols = [
    "escuela_procedencia_raw",
    "dni_raw",
    "fecha_nacimiento_raw",
    "lugar_nacimiento_raw",
    "edad_raw",
    "direccion_raw"
]

for col in cols:

    df_datos_stg[col.replace("_raw", "")] = df_datos_stg[col].apply(
        lambda x: re.sub(r"\s+", " ", str(x).strip())
    )

### Métricas base (inicio de ETAPA 3, aún sin separación)

In [28]:
total_registros = len(df_datos_stg)

telefonos_mult = df_datos_stg["telefono_multiple_flag"].sum()
telefonos_anom = df_datos_stg["telefono_anomalo_flag"].sum()

print("Total:", total_registros)
print("Tel múltiples:", telefonos_mult)
print("Tel anómalos:", telefonos_anom)

Total: 27
Tel múltiples: 2
Tel anómalos: 3


# ETAPA 3 — VALIDACIÓN (Fuente: Datos)

## Objetivo
Evaluar la calidad de `df_datos_stg` y separar los registros en:

- `df_datos_validos`
- `df_datos_revision`
- `reporte_datos`

En esta etapa no se limpia ni se transforma más.
Solo se evalúa si cada registro puede avanzar o debe quedar pendiente de revisión.

### Paso 1 - Crear flags de validación

In [29]:
df_datos_stg["nombre_vacio_flag"] = df_datos_stg["nombre_completo"].eq("")

df_datos_stg["identificador_faltante_flag"] = (
    df_datos_stg["dni_raw"].eq("") &
    df_datos_stg["legajo"].eq("")
&
    
(df_datos_stg["libro_matriz"].eq("") |
 df_datos_stg["folio"].eq(""))
)

df_datos_stg["telefono_dudoso_flag"] = (

df_datos_stg["telefono_num"].ne("") &
    
~df_datos_stg["telefono_largo"].between(8, 11)
)

### Paso 2 - Conteo inicial de flags de validación

Se cuentan los registros marcados por cada flag para conocer la magnitud de los problemas detectados antes de separar válidos y revisión.

In [30]:
conteo_nombre_vacio = df_datos_stg["nombre_vacio_flag"].sum()
conteo_identificador_faltante = df_datos_stg["identificador_faltante_flag"].sum()
conteo_telefono_dudoso = df_datos_stg["telefono_dudoso_flag"].sum()

print("Nombre vacíos:", conteo_nombre_vacio)
print("Identificadores faltantes:", conteo_identificador_faltante)
print("Teléfonos dudosos:", conteo_telefono_dudoso)

Nombre vacíos: 0
Identificadores faltantes: 0
Teléfonos dudosos: 3


### Paso 3 - Separación de registros válidos y en revisión

Criterio:

- Registros válidos:
  - nombre_completo presente
  - al menos un identificador válido

- Registros en revisión:
  - nombre vacío
  - identificadores faltantes

Nota:
Los teléfonos dudosos no bloquean el avance, pero quedan marcados.

In [31]:
condicion_validos = (

~df_datos_stg["nombre_vacio_flag"] &

~df_datos_stg["identificador_faltante_flag"]
)

df_datos_validos = df_datos_stg[condicion_validos].copy()

df_datos_revision = df_datos_stg[~condicion_validos].copy()

print("Válidos:",len(df_datos_validos))
print("Revisión:",len(df_datos_revision))

Válidos: 27
Revisión: 0


### Paso 4 - Reporte de calidad — Fuente Datos

Métricas:

- Total registros
- % registros válidos
- % revisión
- % teléfonos dudosos

In [32]:
total = len(df_datos_stg)
validos = len(df_datos_validos)
revision = len(df_datos_revision)
tel_dudosos = df_datos_stg["telefono_dudoso_flag"].sum()

print("Total:", total)
print("% válidos:",round(validos / total * 100, 2))
print("% revisión:",round(revision / total * 100, 2))
print("% teléfonos dudosos:",round(tel_dudosos / total * 100, 2))

Total: 27
% válidos: 100.0
% revisión: 0.0
% teléfonos dudosos: 11.11


# Cierre de Etapas 1, 2 y 3 — Fuente Datos

Se completó el procesamiento piloto del archivo:

`1°1° 2025 Datos.docx`

## Outputs generados

- `df_datos_raw`
- `df_datos_stg`
- `df_datos_validos`
- `df_datos_revision`
- `reporte_datos`

## Resultado de validación

- Total registros: 27
- Registros válidos: 27
- Registros en revisión: 0
- Teléfonos dudosos: 3

## Conclusión

La fuente Datos del curso 1° división 1° queda procesada correctamente bajo el contrato definido.

Los registros válidos pueden avanzar hacia persistencia intermedia y futura reconciliación de identidad.

In [33]:
conexion = sqlite3.connect("escuela_pipeline.db")

In [34]:
df_datos_validos.to_sql(
    "datos_validos",
    conexion,
    if_exists="replace",
    index=False
)

27

In [35]:
pd.read_sql("SELECT * FROM datos_validos LIMIT 5", conexion)

,nombre_original,libro_matriz_raw,folio_raw,legajo_raw,escuela_procedencia_raw,dni_raw,fecha_nacimiento_raw,lugar_nacimiento_raw,edad_raw,direccion_raw,...,telefono_anomalo_flag,escuela_procedencia,dni,fecha_nacimiento,lugar_nacimiento,edad,direccion,nombre_vacio_flag,identificador_faltante_flag,telefono_dudoso_flag
0,"ACHIGAR, Tiziano",42,240,31/25,25,52746170,26/12/12,Punta Alta,12,Humberto 2538,...,0,25,52746170,26/12/12,Punta Alta,12,Humberto 2538,0,0,0
1,"AGUILAR, Ian Nicolás",42,241,32/25,4,52778569,27/11/12,Punta Alta,12,Bº Albatros XXVII c393,...,0,4,52778569,27/11/12,Punta Alta,12,Bº Albatros XXVII c393,0,0,0
2,"ALTAMIRANO, Joaquin Román",42,242,33/25,8,52887715,14/01/13,San Juan,12,Humberto 1916,...,0,8,52887715,14/01/13,San Juan,12,Humberto 1916,0,0,0
3,"ARANDA, Maia Mariel",42,243,34/25,1,52617713,06/07/12,Punta Alta,12,Entre Ríos 1899,...,0,1,52617713,06/07/12,Punta Alta,12,Entre Ríos 1899,0,0,0
4,"CARRERAS FLAMMINI, Paula Sofía",42,244,35/25,8,52961006,01/02/13,Juris Fed,12,Humberto 2447,...,0,8,52961006,01/02/13,Juris Fed,12,Humberto 2447,0,0,0


### Persistencia intermedia — SQLite (Fuente Datos)

Se implementa almacenamiento en SQLite para persistir los registros válidos de la fuente Datos.

Esto permite:

- Evitar pérdida de información entre ejecuciones
- Facilitar inspección y consulta mediante SQL
- Preparar el pipeline para escalamiento a múltiples cursos/divisiones

Tabla creada:

- `datos_validos`

# ETAPA 1 — EXTRACCIÓN RAW (Fuente: Legajos)

## Objetivo
Construir `df_legajos_raw` preservando el contenido del archivo Word tal como viene en origen.

## Archivo piloto
`1°1° 2025 Legajos.docx`

## Entrada
- Documento Word (.docx)
- Tabla con información de alumnos (legajos)

## Proceso
1. Abrir el documento
2. Identificar la tabla
3. Recorrer filas (excluyendo encabezado)
4. Extraer valores por celda
5. Agregar metadatos de contexto:
   - curso
   - division
   - anio
   - fuente
   - archivo_origen

## Regla metodológica

En esta etapa:

✔ preservar datos originales  
✔ capturar campos `_raw`  
✔ no limpiar  
✔ no transformar  
✔ no validar  

## Output esperado

DataFrame:

`df_legajos_raw`

Columnas:

- nombre_original  
- libro_matriz_raw  
- folio_raw  
- legajo_raw  
- curso  
- division  
- anio  
- fuente  
- archivo_origen

In [36]:
ruta_legajos = r"C:\Users\amilc\OneDrive\Proyecto Data Science Media 2\EES Nº 2 2025-20260117T072850Z-3-001\EES Nº 2 2025\Datos 2025\Legajos\1°1° 2025 Legajos.docx"

doc_legajos = Document(ruta_legajos)

In [37]:
len(doc_legajos.tables)

1

In [38]:
tabla_legajos = doc_legajos.tables[0]

In [39]:
len(tabla_legajos.rows), len(tabla_legajos.rows[0].cells)

(28, 5)

In [40]:
registros_legajos_raw = []

In [41]:
for i, fila in enumerate(tabla_legajos.rows[:6]): 
    print("FILA:",i)
    print(fila.cells[0].text)
    print(fila.cells[1].text)
    print(fila.cells[2].text)
    print(fila.cells[3].text)
    print(fila.cells[4].text)
    print("-----")

FILA: 0
Nº
APELLIDO Y NOMBRES DEL ALUMNO
LM
F
Leg
-----
FILA: 1

 ACHIGAR, Tiziano
42
240
31/25
-----
FILA: 2

AGUILAR, Ian Nicolás
42
241
32/25
-----
FILA: 3

ALTAMIRANO, Joaquin Román
42
242
33/25
-----
FILA: 4

ARANDA, Maia Mariel
42
243
34/25
-----
FILA: 5

CARRERAS FLAMMINI, Paula Sofía
42
244
35/25
-----


In [42]:
registros_legajos_raw = []

for fila in tabla_legajos.rows[1:]:
    registro = {
        "nombre_original": fila.cells[1].text,
        "libro_matriz_raw": fila.cells[2].text,
        "folio_raw": fila.cells[3].text,
        "legajo_raw": fila.cells[4].text,
        "curso": "1",
        "division": "1",
        "anio": "2025",
        "fuente": "Legajos",
        "archivo_origen": "1°1° 2025 Legajos.docx"
    }
    registros_legajos_raw.append(registro)

df_legajos_raw = pd.DataFrame(registros_legajos_raw)

In [43]:
df_legajos_raw.head()

,nombre_original,libro_matriz_raw,folio_raw,legajo_raw,curso,division,anio,fuente,archivo_origen
0,"ACHIGAR, Tiziano",42,240,31/25,1,1,2025,Legajos,1°1° 2025 Legajos.docx
1,"AGUILAR, Ian Nicolás",42,241,32/25,1,1,2025,Legajos,1°1° 2025 Legajos.docx
2,"ALTAMIRANO, Joaquin Román",42,242,33/25,1,1,2025,Legajos,1°1° 2025 Legajos.docx
3,"ARANDA, Maia Mariel",42,243,34/25,1,1,2025,Legajos,1°1° 2025 Legajos.docx
4,"CARRERAS FLAMMINI, Paula Sofía",42,244,35/25,1,1,2025,Legajos,1°1° 2025 Legajos.docx


# ETAPA 2 — STAGING (Fuente: Legajos)

## Objetivo
Transformar `df_legajos_raw` en una estructura consistente (`df_legajos_stg`) lista para validación y reconciliación.

## Subetapas

### A. Normalización técnica
- eliminar espacios externos (`strip`)
- colapsar espacios múltiples
- normalizar texto

### B. Transformación estructural
- renombrar campos
- preparar columnas para matching

## Reglas

- no perder información original
- mantener columnas `_raw`
- crear nuevas columnas limpias

## Output esperado

`df_legajos_stg`

Columnas principales:

- nombre_completo
- libro_matriz
- folio
- legajo
- + columnas `_raw`
- + contexto (curso, division, anio, fuente, archivo_origen)

In [44]:
df_legajos_stg = df_legajos_raw.copy()

### Normalización de los nombres de alumnos

In [45]:
df_legajos_stg["nombre_completo"] = df_legajos_stg["nombre_original"].apply(
    lambda x: re.sub(r'\s+', ' ', x.strip())
)

### Normalización de identificadores administrativos

Se construyen versiones normalizadas de:

- libro_matriz
- folio
- legajo

Preservando sus campos `_raw` originales.

Regla:
Estos campos se tratan como identificadores administrativos (texto),
no como variables numéricas para cálculo.

In [46]:
df_legajos_stg["libro_matriz"] = df_legajos_stg["libro_matriz_raw"]
df_legajos_stg["folio"] = df_legajos_stg["folio_raw"]
df_legajos_stg["legajo"] = df_legajos_stg["legajo_raw"]

In [47]:
df_legajos_stg[[
    "nombre_original",
    "nombre_completo",
    "libro_matriz_raw",
    "libro_matriz",
    "folio_raw",
    "folio",
    "legajo_raw",
    "legajo"
]].head()

,nombre_original,nombre_completo,libro_matriz_raw,libro_matriz,folio_raw,folio,legajo_raw,legajo
0,"ACHIGAR, Tiziano","ACHIGAR, Tiziano",42,42,240,240,31/25,31/25
1,"AGUILAR, Ian Nicolás","AGUILAR, Ian Nicolás",42,42,241,241,32/25,32/25
2,"ALTAMIRANO, Joaquin Román","ALTAMIRANO, Joaquin Román",42,42,242,242,33/25,33/25
3,"ARANDA, Maia Mariel","ARANDA, Maia Mariel",42,42,243,243,34/25,34/25
4,"CARRERAS FLAMMINI, Paula Sofía","CARRERAS FLAMMINI, Paula Sofía",42,42,244,244,35/25,35/25


# ETAPA 3 — VALIDACIÓN (Fuente: Legajos)

## Objetivo
Evaluar la calidad de `df_legajos_stg` y separar los registros en:

- `df_legajos_validos`
- `df_legajos_revision`
- `reporte_legajos`

## Reglas de validación

Un registro es válido si:

- nombre_completo NO está vacío
- Y tiene al menos:
  - legajo
  - o (libro_matriz + folio)

Los duplicados se marcan pero no bloquean avance (requieren revisión posterior).

### Paso 1 - Crear flags de validación

In [48]:
df_legajos_stg["nombre_vacio_flag"] = df_legajos_stg["nombre_completo"].eq("")

df_legajos_stg["identificador_faltante_flag"] = (
    df_legajos_stg["legajo"].eq("") &
    (df_legajos_stg["libro_matriz"].eq("") | df_legajos_stg["folio"].eq(""))
)

### Paso 2 - Identificación de duplicados

In [49]:
df_legajos_stg["duplicado_nombre_flag"] = df_legajos_stg["nombre_completo"].duplicated(keep=False)

df_legajos_stg["duplicado_legajo_flag"] = df_legajos_stg["legajo"].duplicated(keep=False)

### Paso 3 - Conteo inicial de flags de validación

Se cuentan los registros marcados por cada flag para conocer la magnitud de los problemas detectados antes de separar válidos y revisión.

In [50]:
print("Nombre vacío:", df_legajos_stg["nombre_vacio_flag"].sum())
print("Identificador faltante:", df_legajos_stg["identificador_faltante_flag"].sum())
print("Duplicados nombre:", df_legajos_stg["duplicado_nombre_flag"].sum())
print("Duplicados legajo:", df_legajos_stg["duplicado_legajo_flag"].sum())

Nombre vacío: 0
Identificador faltante: 0
Duplicados nombre: 0
Duplicados legajo: 0


### Paso 4 - Separación de registros válidos y en revisión

Criterio:

- Registros válidos:
  - nombre_completo presente
  - al menos un identificador válido

- Registros en revisión:
  - nombre vacío
  - identificadores faltantes

Nota:
Los teléfonos dudosos no bloquean el avance, pero quedan marcados.

In [51]:
condicion_validos = (
    ~df_legajos_stg["nombre_vacio_flag"] &
    ~df_legajos_stg["identificador_faltante_flag"]
)

df_legajos_validos = df_legajos_stg[condicion_validos].copy()

df_legajos_revision = df_legajos_stg[~condicion_validos].copy()

### Paso 5 - Chequeo de registros 

In [52]:
print("Válidos:", len(df_legajos_validos))
print("Revisión:", len(df_legajos_revision))
print("Duplicados nombre:", df_legajos_stg["duplicado_nombre_flag"].sum())
print("Duplicados legajo:", df_legajos_stg["duplicado_legajo_flag"].sum())

Válidos: 27
Revisión: 0
Duplicados nombre: 0
Duplicados legajo: 0


### Paso 6 - Reporte de calidad — Fuente Legajos

Métricas:

- Total registros
- % registros válidos
- % revisión
- % nombres duplicados
- % legajos duplicados

In [53]:
total = len(df_legajos_stg)
validos = len(df_legajos_validos)
revision = len(df_legajos_revision)
nom_duplicados = df_legajos_stg["duplicado_nombre_flag"].sum()
leg_duplicados = df_legajos_stg["duplicado_legajo_flag"].sum()

print("Total:", total)
print("% válidos:",round(validos / total * 100, 2))
print("% revisión:",round(revision / total * 100, 2))
print("% nombres duplicados:",round(nom_duplicados / total * 100, 2))
print("% legajos duplicados:",round(leg_duplicados / total * 100, 2))

Total: 27
% válidos: 100.0
% revisión: 0.0
% nombres duplicados: 0.0
% legajos duplicados: 0.0


## Cierre ETAPAS 1, 2 y 3 — Fuente Legajos

- Total registros: 27  
- Registros válidos: 27  
- Registros en revisión: 0  

Conclusión:
Fuente consistente, sin problemas estructurales.
Apta para proceso de reconciliación en Etapa 4.

In [54]:
df_legajos_validos.to_sql(
    "legajos_validos",
    conexion,
    if_exists="replace",
    index=False
)

27

In [55]:
pd.read_sql("SELECT * FROM legajos_validos LIMIT 5", conexion)

,nombre_original,libro_matriz_raw,folio_raw,legajo_raw,curso,division,anio,fuente,archivo_origen,nombre_completo,libro_matriz,folio,legajo,nombre_vacio_flag,identificador_faltante_flag,duplicado_nombre_flag,duplicado_legajo_flag
0,"ACHIGAR, Tiziano",42,240,31/25,1,1,2025,Legajos,1°1° 2025 Legajos.docx,"ACHIGAR, Tiziano",42,240,31/25,0,0,0,0
1,"AGUILAR, Ian Nicolás",42,241,32/25,1,1,2025,Legajos,1°1° 2025 Legajos.docx,"AGUILAR, Ian Nicolás",42,241,32/25,0,0,0,0
2,"ALTAMIRANO, Joaquin Román",42,242,33/25,1,1,2025,Legajos,1°1° 2025 Legajos.docx,"ALTAMIRANO, Joaquin Román",42,242,33/25,0,0,0,0
3,"ARANDA, Maia Mariel",42,243,34/25,1,1,2025,Legajos,1°1° 2025 Legajos.docx,"ARANDA, Maia Mariel",42,243,34/25,0,0,0,0
4,"CARRERAS FLAMMINI, Paula Sofía",42,244,35/25,1,1,2025,Legajos,1°1° 2025 Legajos.docx,"CARRERAS FLAMMINI, Paula Sofía",42,244,35/25,0,0,0,0


### Persistencia intermedia — SQLite (Fuente Legajos)

Se persisten los registros válidos de la fuente Legajos en SQLite para mantener trazabilidad y preparar la reconciliación contra Datos.

# ETAPA 4 — RECONCILIACIÓN (Datos vs Legajos)

## Objetivo
Comparar registros entre las fuentes:

- Datos
- Legajos

para verificar consistencia de identidad antes de construir `df_identidad`.

## Estrategia inicial

Cruce por:

- nombre_completo

Validando:

- coincidencias
- faltantes
- inconsistencias

## Inputs

- df_datos_validos
- df_legajos_validos

### Paso 1 - Creación de conjuntos de nombres de ambas fuentes

In [56]:
nombres_datos = set(df_datos_validos["nombre_completo"])

nombres_legajos = set(df_legajos_validos["nombre_completo"])

### Paso 2 - Alumnos que están en una fuente y no en la otra

In [57]:
solo_datos = nombres_datos - nombres_legajos

solo_legajos = nombres_legajos - nombres_datos

In [58]:
print("Sólo en Datos:", len(solo_datos))
print("Sólo en Legajos:", len(solo_legajos))

Sólo en Datos: 0
Sólo en Legajos: 0


### Paso 3 — Merge de reconciliación

Se realiza un cruce entre:

- df_datos_validos
- df_legajos_validos

usando:

`nombre_completo`

Objetivo:

- consolidar identidad
- verificar integridad entre fuentes
- preparar futura generación de `id_persona`

In [59]:
df_identidad = df_datos_validos.merge(
    df_legajos_validos,
    on="nombre_completo",
    how="inner",
    suffixes=("_datos","_legajos")
)

In [60]:
len(df_identidad)

27

In [61]:
df_identidad.head()

,nombre_original_datos,libro_matriz_raw_datos,folio_raw_datos,legajo_raw_datos,escuela_procedencia_raw,dni_raw,fecha_nacimiento_raw,lugar_nacimiento_raw,edad_raw,direccion_raw,...,anio_legajos,fuente_legajos,archivo_origen_legajos,libro_matriz_legajos,folio_legajos,legajo_legajos,nombre_vacio_flag_legajos,identificador_faltante_flag_legajos,duplicado_nombre_flag,duplicado_legajo_flag
0,"ACHIGAR, Tiziano",42,240,31/25,25,52746170,26/12/12,Punta Alta,12,Humberto 2538,...,2025,Legajos,1°1° 2025 Legajos.docx,42,240,31/25,False,False,False,False
1,"AGUILAR, Ian Nicolás",42,241,32/25,4,52778569,27/11/12,Punta Alta,12,Bº Albatros XXVII c393,...,2025,Legajos,1°1° 2025 Legajos.docx,42,241,32/25,False,False,False,False
2,"ALTAMIRANO, Joaquin Román",42,242,33/25,8,52887715,14/01/13,San Juan,12,Humberto 1916,...,2025,Legajos,1°1° 2025 Legajos.docx,42,242,33/25,False,False,False,False
3,"ARANDA, Maia Mariel",42,243,34/25,1,52617713,06/07/12,Punta Alta,12,Entre Ríos 1899,...,2025,Legajos,1°1° 2025 Legajos.docx,42,243,34/25,False,False,False,False
4,"CARRERAS FLAMMINI, Paula Sofía",42,244,35/25,8,52961006,01/02/13,Juris Fed,12,Humberto 2447,...,2025,Legajos,1°1° 2025 Legajos.docx,42,244,35/25,False,False,False,False


### Paso 4 — Consolidación de columnas

Luego del merge se consolidan columnas equivalentes entre fuentes.

Criterios:

- Datos domina atributos personales
- Legajos domina identificadores administrativos
- Campos redundantes se unifican

#### Paso 4.1 - Consolidación de contexto

In [62]:
df_identidad["curso"] = df_identidad["curso_datos"]

df_identidad["division"] = df_identidad["division_datos"]

df_identidad["anio"] = df_identidad["anio_datos"]

#### Paso 4.2 - Consolidación de identificadores administrativos

In [63]:
df_identidad["libro_matriz"] = df_identidad["libro_matriz_legajos"]

df_identidad["folio"] = df_identidad["folio_legajos"]

df_identidad["legajo"] = df_identidad["legajo_legajos"]

#### Paso 4.3 - Consolidación de identificadores personales

In [64]:
df_identidad["dni"] = df_identidad["dni_raw"]

df_identidad["fecha_nacimiento"] = df_identidad["fecha_nacimiento_raw"]

### Paso 5 — Verificación de consistencia entre fuentes

Se comparan columnas equivalentes provenientes de:

- Datos
- Legajos

Objetivo:

detectar inconsistencias antes de eliminar redundancia.

#### Paso 5.1 - Verificación de curso

In [65]:
(df_identidad["curso_datos"] == df_identidad["curso_legajos"]).value_counts()

True    27
Name: count, dtype: int64

#### Paso 5.2 - Verificación de división

In [66]:
(df_identidad["division_datos"] == df_identidad["division_legajos"]).value_counts()

True    27
Name: count, dtype: int64

#### Paso 5.3 - Verificación de año

In [67]:
(df_identidad["anio_datos"] == df_identidad["anio_legajos"]).value_counts()

True    27
Name: count, dtype: int64

### Paso 6 — Limpieza de redundancia post-reconciliación

Luego de validar consistencia entre fuentes, se eliminan columnas redundantes provenientes del merge.

Objetivo:

- simplificar `df_identidad`
- conservar únicamente columnas maestras consolidadas
- mantener trazabilidad lógica del pipeline

In [68]:
df_identidad.columns.tolist()

['nombre_original_datos',
 'libro_matriz_raw_datos',
 'folio_raw_datos',
 'legajo_raw_datos',
 'escuela_procedencia_raw',
 'dni_raw',
 'fecha_nacimiento_raw',
 'lugar_nacimiento_raw',
 'edad_raw',
 'direccion_raw',
 'telefono_raw',
 'curso_datos',
 'division_datos',
 'anio_datos',
 'fuente_datos',
 'archivo_origen_datos',
 'nombre_completo',
 'libro_matriz_datos',
 'folio_datos',
 'legajo_datos',
 'telefono_num',
 'tipo_contacto',
 'telefono_multiple_flag',
 'telefono_largo',
 'telefono_anomalo_flag',
 'escuela_procedencia',
 'dni',
 'fecha_nacimiento',
 'lugar_nacimiento',
 'edad',
 'direccion',
 'nombre_vacio_flag_datos',
 'identificador_faltante_flag_datos',
 'telefono_dudoso_flag',
 'nombre_original_legajos',
 'libro_matriz_raw_legajos',
 'folio_raw_legajos',
 'legajo_raw_legajos',
 'curso_legajos',
 'division_legajos',
 'anio_legajos',
 'fuente_legajos',
 'archivo_origen_legajos',
 'libro_matriz_legajos',
 'folio_legajos',
 'legajo_legajos',
 'nombre_vacio_flag_legajos',
 'identif

### Paso 7 — Creación de `df_identidad_limpia`

A partir de `df_identidad`, se construye una versión depurada con columnas consolidadas.

Criterios:

- Se conservan atributos de identidad y contexto académico.
- Se conservan identificadores administrativos consolidados.
- Se conservan columnas de teléfono porque todavía contienen casos abiertos de calidad.
- Se eliminan columnas `_raw`, duplicadas por fuente y flags que ya cumplieron su función en etapas anteriores.

La trazabilidad completa queda preservada en:

- `df_datos_raw`
- `df_datos_stg`
- `df_legajos_raw`
- `df_legajos_stg`
- SQLite

In [69]:
columnas_identidad = [
    "nombre_completo",
    "dni",
    "fecha_nacimiento",
    "lugar_nacimiento",
    "edad",
    "direccion",
    "escuela_procedencia",
    "telefono_raw",
    "telefono_num",
    "tipo_contacto",
    "telefono_multiple_flag",
    "telefono_largo",
    "telefono_anomalo_flag",
    "curso",
    "division",
    "anio",
    "libro_matriz",
    "folio",
    "legajo"
]

df_identidad_limpia = df_identidad[columnas_identidad].copy()

In [70]:
df_identidad_limpia.head()

,nombre_completo,dni,fecha_nacimiento,lugar_nacimiento,edad,direccion,escuela_procedencia,telefono_raw,telefono_num,tipo_contacto,telefono_multiple_flag,telefono_largo,telefono_anomalo_flag,curso,division,anio,libro_matriz,folio,legajo
0,"ACHIGAR, Tiziano",52746170,26/12/12,Punta Alta,12,Humberto 2538,25,2932-520752M,2932520752,M,False,10,False,1,1,2025,42,240,31/25
1,"AGUILAR, Ian Nicolás",52778569,27/11/12,Punta Alta,12,Bº Albatros XXVII c393,4,2932-573441M,2932573441,M,False,10,False,1,1,2025,42,241,32/25
2,"ALTAMIRANO, Joaquin Román",52887715,14/01/13,San Juan,12,Humberto 1916,8,2932-614538M,2932614538,M,False,10,False,1,1,2025,42,242,33/25
3,"ARANDA, Maia Mariel",52617713,06/07/12,Punta Alta,12,Entre Ríos 1899,1,2932-520048P,2932520048,P,False,10,False,1,1,2025,42,243,34/25
4,"CARRERAS FLAMMINI, Paula Sofía",52961006,01/02/13,Juris Fed,12,Humberto 2447,8,2932-505154M,2932505154,M,False,10,False,1,1,2025,42,244,35/25


In [71]:
df_identidad_limpia.shape

(27, 19)

### Paso 8 — Etapa 4C: Generación de `id_persona`

Se genera `id_persona` como surrogate key interna y estable para cada alumno identificado, a través de una estrategia acumulativa, generando la clave en base al último `id_persona` existente en SQLite. Este método tiene por objetivo:

- evitar colisiones de ID al procesar múltiples cursos/divisiones
- permitir persistencia incremental en SQLite
- preparar el pipeline para modularización y escalamiento

La clave `id_persona` permite reemplazar progresivamente `nombre_completo` como vínculo provisional entre entidades.

A partir de este punto, `id_persona` debe propagarse hacia:

- `df_alumnos_master`
- `df_contactos`
- futuras tablas relacionales

In [72]:
def obtener_ultimo_id_persona(conexion):

    try:
        resultado = pd.read_sql(
            """
            SELECT MAX(id_persona) AS ultimo_id
            FROM identidad_limpia
            """,
            conexion
        )

        ultimo_id = resultado.loc[0, "ultimo_id"]

        if pd.isna(ultimo_id):
            return 0

        return int(ultimo_id)

    except Exception as e:
        print("ERROR al obtener ultimo_id_persona:")
        print(e)
        raise

In [73]:
df_identidad_limpia = df_identidad_limpia.reset_index(drop=True)

ultimo_id = obtener_ultimo_id_persona(conexion)

df_identidad_limpia["id_persona"] = range(
    ultimo_id + 1,
    ultimo_id + 1 + len(df_identidad_limpia)
)

In [74]:
df_identidad_limpia[["id_persona", "nombre_completo"]].head()

,id_persona,nombre_completo
0,55,"ACHIGAR, Tiziano"
1,56,"AGUILAR, Ian Nicolás"
2,57,"ALTAMIRANO, Joaquin Román"
3,58,"ARANDA, Maia Mariel"
4,59,"CARRERAS FLAMMINI, Paula Sofía"


### Paso 9 — Separación de contactos del master de alumnos

Se separa la información telefónica de `df_identidad_limpia` para evitar que el futuro `df_alumnos_master` contenga campos multivaluados o datos de contacto pendientes de normalización.

Decisión de diseño:

- `df_alumnos_master` conserva datos propios de identidad, contexto académico e identificadores administrativos.
- `df_contactos` conserva información telefónica y flags de calidad asociados.

Motivo:

Un alumno puede tener cero, uno o múltiples teléfonos.  
Por lo tanto, los teléfonos no deben quedar como columnas fijas dentro del master de alumnos.

Outputs esperados:

- `df_contactos`
- `df_alumnos_master`

#### Paso 9.1 - Creación de df_contactos

In [75]:
df_contactos = df_identidad_limpia[
    [
        "id_persona",
        "nombre_completo",
        "telefono_raw",
        "telefono_num",
        "tipo_contacto",
        "telefono_multiple_flag",
        "telefono_largo",
        "telefono_anomalo_flag"
    ]
    ].copy()

#### Paso 9.2 - Quitar columnas de teléfonos del master de alumnos

In [76]:
columnas_master = [
    "id_persona",
    "nombre_completo",
    "dni",
    "fecha_nacimiento",
    "lugar_nacimiento",
    "edad",
    "direccion",
    "escuela_procedencia",
    "curso",
    "division",
    "anio",
    "libro_matriz",
    "folio",
    "legajo"
]

df_alumnos_master = df_identidad_limpia[columnas_master].copy()

# Cierre ETAPA 4 — Reconciliación y consolidación de identidad

## Objetivo cumplido

Se completó la reconciliación entre las fuentes:

- Datos
- Legajos

para construir una primera entidad consolidada de alumnos con identidad persistente.

---

## Procesos realizados

### 4A — Reconciliación inicial

Se verificó coincidencia completa de identidades entre ambas fuentes utilizando:

- `nombre_completo`

Resultado:

- sin faltantes
- sin sobrantes
- sin inconsistencias nominales

---

### Merge de consolidación

Se construyó:

- `df_identidad`

mediante integración de ambas fuentes.

---

### Consolidación de columnas

Se unificaron atributos equivalentes:

- contexto académico
- identificadores administrativos
- atributos personales

definiendo fuentes dominantes según tipo de dato.

---

### Verificación post-merge

Se validó consistencia entre:

- curso
- división
- año

detectando y corrigiendo una inconsistencia de tipo de dato (`int` vs `string`) en `anio`.

La corrección se realizó en origen para preservar coherencia del pipeline.

---

## 4B — Consolidación post-reconciliación

### Limpieza de redundancia

Se eliminaron:

- columnas duplicadas
- metadata temporal de merge
- flags ya resueltos
- redundancia entre fuentes

Resultado:

- `df_identidad_limpia`

---

### Separación de entidades multivaluadas

Se separó la información telefónica hacia:

- `df_contactos`

para evitar estructuras multivaluadas dentro de:

- `df_alumnos_master`

---

### Construcción de master

Se generó:

- `df_alumnos_master`

como primera entidad consolidada de alumnos.

---

## 4C — Generación de identidad persistente

Se generó:

- `id_persona`

como surrogate key interna y estable para cada alumno reconciliado.

Objetivos:

- independizar relaciones del uso de nombre o DNI
- preparar futuras relaciones SQL
- permitir referencias consistentes entre entidades

A partir de esta etapa:

- Contactos
- Movimientos
- futuras entidades relacionales

deben vincularse mediante `id_persona`.

---

## Resultado general

La ETAPA 4 finaliza con:

- identidad reconciliada
- identidad persistente (`id_persona`)
- estructura relacional inicial
- redundancia controlada
- separación conceptual entre alumnos y contactos
- preparación para relaciones SQL futuras

El pipeline queda preparado para:

- normalización final de Contactos
- integración de Eventos / Marzo
- expansión a múltiples cursos y entidades

# Contrato — Entidad Contactos

## Objetivo
Construir una tabla derivada de contactos telefónicos a partir de `df_contactos`, separada del master de alumnos.

## Granularidad
Una fila representa un teléfono asociado a un alumno.

Por lo tanto:

- un alumno puede aparecer una vez si tiene un teléfono
- un alumno puede aparecer varias veces si tiene múltiples teléfonos
- un alumno puede no aparecer si no tiene teléfono registrado

## Input
`df_contactos`

Derivado de:

`df_identidad_limpia`

## Outputs
- `df_contactos_normalizados`
- `df_contactos_validos`
- `df_contactos_revision`

## Reglas de transformación
- Si `telefono_raw` contiene múltiples teléfonos separados por `/`, se expande a varias filas.
- Se conserva `telefono_raw` como evidencia.
- Se conserva `tipo_contacto` si está disponible.
- Se normaliza `telefono_num` como texto.
- No se completan prefijos ni partes faltantes de teléfonos.

## Reglas de validación
Un contacto válido debe:

- tener `telefono_num`
- no estar marcado como incompleto
- no estar marcado como anómalo

Un contacto va a revisión si:

- `telefono_incompleto_flag = True`
- o `telefono_anomalo_flag = True`

## Decisión de diseño
Los teléfonos no forman parte de `df_alumnos_master` porque representan una entidad multivaluada.

La relación correcta es:

`Alumno 1 → N Contactos`

## Nota futura
Cuando exista `id_persona`, `df_contactos` deberá usar `id_persona` como clave de relación en lugar de `nombre_completo`.

# ETAPA 5 — Normalización de contactos

## Objetivo

Revisar y preparar `df_contactos` para futura normalización relacional.

Problemas presentes:

- teléfonos múltiples
- formatos heterogéneos
- tipos de contacto mezclados
- registros anómalos

## Objetivo técnico

Separar:

- contactos válidos simples
- contactos múltiples
- contactos con anomalías

para futura expansión y limpieza.

In [77]:
df_contactos.head(10)

,id_persona,nombre_completo,telefono_raw,telefono_num,tipo_contacto,telefono_multiple_flag,telefono_largo,telefono_anomalo_flag
0,55,"ACHIGAR, Tiziano",2932-520752M,2932520752,M,False,10,False
1,56,"AGUILAR, Ian Nicolás",2932-573441M,2932573441,M,False,10,False
2,57,"ALTAMIRANO, Joaquin Román",2932-614538M,2932614538,M,False,10,False
3,58,"ARANDA, Maia Mariel",2932-520048P,2932520048,P,False,10,False
4,59,"CARRERAS FLAMMINI, Paula Sofía",2932-505154M,2932505154,M,False,10,False
5,60,"CEMINO, Tomas Agustín",2932-410841M,2932410841,M,False,10,False
6,61,"D`ACHILLI, Juana Antonella",2932474863/441518M,2932474863441518,M,True,16,True
7,62,"DUHAU, Oriana Marina",2932-404188M,2932404188,M,False,10,False
8,63,"ESPINOZA, Gael Mateo",2932-447160/441779M,2932447160441779,M,True,16,True
9,64,"FALASCHI, Oriana Yanil",2932-635639M,2932635639,M,False,10,False


In [78]:
df_contactos.columns.tolist()

['id_persona',
 'nombre_completo',
 'telefono_raw',
 'telefono_num',
 'tipo_contacto',
 'telefono_multiple_flag',
 'telefono_largo',
 'telefono_anomalo_flag']

### Paso 1 — Separación de contactos múltiples

Se identifican registros con múltiples teléfonos en un mismo campo.

Objetivo:

transformar estructuras multivaluadas en múltiples filas simples para futura normalización relacional.

Resultado esperado:

- un teléfono por fila
- mantenimiento de relación con el alumno
- conservación del tipo de contacto

In [79]:
df_contactos_multiples = df_contactos[
    df_contactos["telefono_multiple_flag"]
    ].copy()

In [80]:
df_contactos_multiples

,id_persona,nombre_completo,telefono_raw,telefono_num,tipo_contacto,telefono_multiple_flag,telefono_largo,telefono_anomalo_flag
6,61,"D`ACHILLI, Juana Antonella",2932474863/441518M,2932474863441518,M,True,16,True
8,63,"ESPINOZA, Gael Mateo",2932-447160/441779M,2932447160441779,M,True,16,True


## Paso 2 — Preparación de expansión de teléfonos múltiples

Se preparan registros multivaluados para futura expansión relacional.

Proceso:

- separación del tipo de contacto
- identificación de delimitadores
- preparación de listas de teléfonos

Objetivo:

obtener un teléfono individual por fila.

In [81]:
df_contactos_multiples["telefonos_lista"] = (
    df_contactos_multiples["telefono_raw"]
    .str[:-1]
    .str.split("/")
)

In [82]:
df_contactos_multiples[
    ["id_persona",
     "nombre_completo",
    "telefono_raw",
    "telefonos_lista"]
    ]

,id_persona,nombre_completo,telefono_raw,telefonos_lista
6,61,"D`ACHILLI, Juana Antonella",2932474863/441518M,"[2932474863, 441518]"
8,63,"ESPINOZA, Gael Mateo",2932-447160/441779M,"[2932-447160, 441779]"


### Paso 3 — Expansión de teléfonos múltiples

Se convierte cada lista de teléfonos en filas individuales.

Objetivo:

- un teléfono por fila
- conservar el alumno
- conservar tipo_contacto

In [83]:
df_contactos_multiples_exp = df_contactos_multiples.explode("telefonos_lista").copy()

In [84]:
df_contactos_multiples_exp["telefono_num"] = df_contactos_multiples_exp["telefonos_lista"]

In [85]:
df_contactos_multiples_exp[
    ["id_persona",
     "nombre_completo",
     "telefono_raw",
     "telefono_num",
     "tipo_contacto"]
    ]

,id_persona,nombre_completo,telefono_raw,telefono_num,tipo_contacto
6,61,"D`ACHILLI, Juana Antonella",2932474863/441518M,2932474863,M
6,61,"D`ACHILLI, Juana Antonella",2932474863/441518M,441518,M
8,63,"ESPINOZA, Gael Mateo",2932-447160/441779M,2932-447160,M
8,63,"ESPINOZA, Gael Mateo",2932-447160/441779M,441779,M


### Paso 4 — Consolidación de contactos normalizados

Se unifican:

- contactos simples
- contactos múltiples expandidos

para construir una única tabla de contactos normalizados.

Objetivo:

obtener una estructura final con:

- un teléfono por fila
- un alumno por registro de contacto
- flags preservados

#### Subpaso 4.1 - Contactos simples

In [86]:
df_contactos_simples = df_contactos[
    ~df_contactos["telefono_multiple_flag"]
    ].copy()

#### Subpaso 4.2 - Unificación

In [87]:
df_contactos_normalizados = pd.concat(
    [
        df_contactos_simples,
        df_contactos_multiples_exp
    ],
    ignore_index=True
)

In [88]:
len(df_contactos_normalizados)

29

In [89]:
df_contactos_normalizados[
    [
        "id_persona",
        "nombre_completo",
        "telefono_num",
        "tipo_contacto",
        "telefono_multiple_flag"
    ]
    ].tail(10)

,id_persona,nombre_completo,telefono_num,tipo_contacto,telefono_multiple_flag
19,76,"SANCHEZ PEÑA, Daiana Jazmín",2932573334,M,False
20,77,"SANCHEZ PEÑA, Milagros Agostina",2932573334,M,False
21,78,"UBEDA, Franco Lionel",2932509256,M,False
22,79,"UÑATE, Thiago Santino",2932631139,M,False
23,80,"ZACARIAS, Kevin Alejandro",2932457126,M,False
24,81,"ZERPA, Fernando Damián",2932476473,P,False
25,61,"D`ACHILLI, Juana Antonella",2932474863,M,True
26,61,"D`ACHILLI, Juana Antonella",441518,M,True
27,63,"ESPINOZA, Gael Mateo",2932-447160,M,True
28,63,"ESPINOZA, Gael Mateo",441779,M,True


### Paso 5 — Limpieza post-normalización de contactos

Se eliminan columnas temporales utilizadas durante la expansión de teléfonos múltiples.

También se normalizan valores finales de teléfono.

Objetivo:

obtener una tabla final de contactos normalizados.

#### Subpaso 5.1 - Limpiar espacios

In [90]:
df_contactos_normalizados["telefono_num"] = (
    df_contactos_normalizados["telefono_num"]
    .astype(str)
    .str.strip()
)

#### Subpaso 5.2 - Eliminar columna temporal

In [91]:
df_contactos_normalizados = df_contactos_normalizados.drop(
    columns=["telefonos_lista"],
    errors="ignore"
)

In [92]:
df_contactos_normalizados.columns.tolist()

['id_persona',
 'nombre_completo',
 'telefono_raw',
 'telefono_num',
 'tipo_contacto',
 'telefono_multiple_flag',
 'telefono_largo',
 'telefono_anomalo_flag']

In [93]:
df_contactos_normalizados.tail(10)

,id_persona,nombre_completo,telefono_raw,telefono_num,tipo_contacto,telefono_multiple_flag,telefono_largo,telefono_anomalo_flag
19,76,"SANCHEZ PEÑA, Daiana Jazmín",2932-573334M,2932573334,M,False,10,False
20,77,"SANCHEZ PEÑA, Milagros Agostina",2932-573334M,2932573334,M,False,10,False
21,78,"UBEDA, Franco Lionel",2932-509256M,2932509256,M,False,10,False
22,79,"UÑATE, Thiago Santino",2932-631139M,2932631139,M,False,10,False
23,80,"ZACARIAS, Kevin Alejandro",2932-457126M,2932457126,M,False,10,False
24,81,"ZERPA, Fernando Damián",2932-476473P,2932476473,P,False,10,False
25,61,"D`ACHILLI, Juana Antonella",2932474863/441518M,2932474863,M,True,16,True
26,61,"D`ACHILLI, Juana Antonella",2932474863/441518M,441518,M,True,16,True
27,63,"ESPINOZA, Gael Mateo",2932-447160/441779M,2932-447160,M,True,16,True
28,63,"ESPINOZA, Gael Mateo",2932-447160/441779M,441779,M,True,16,True


### Paso 6 — Clasificación de teléfonos incompletos

Luego de la expansión de contactos múltiples, se detectan teléfonos con longitud insuficiente.

Estos registros no se corrigen automáticamente para evitar fabricación de datos.

Objetivo:

identificar contactos incompletos para futura revisión manual o reconciliación con nuevas fuentes.

In [94]:
df_contactos_normalizados["telefono_incompleto_flag"] = (
    df_contactos_normalizados["telefono_num"]
    .astype(str)
    .str.len() < 10
)

In [95]:
df_contactos_normalizados[
    df_contactos_normalizados["telefono_incompleto_flag"]
    ]

,id_persona,nombre_completo,telefono_raw,telefono_num,tipo_contacto,telefono_multiple_flag,telefono_largo,telefono_anomalo_flag,telefono_incompleto_flag
26,61,"D`ACHILLI, Juana Antonella",2932474863/441518M,441518,M,True,16,True,True
28,63,"ESPINOZA, Gael Mateo",2932-447160/441779M,441779,M,True,16,True,True


### Paso 7 — Separación de contactos válidos y revisión

Se clasifican los contactos normalizados según calidad del número telefónico.

Objetivo:

separar:

- contactos válidos
- contactos incompletos o anómalos

para futura revisión o reconciliación manual.

In [96]:
condicion_contacto_valido = (
    ~df_contactos_normalizados["telefono_incompleto_flag"] &
    ~df_contactos_normalizados["telefono_anomalo_flag"]
)

In [97]:
df_contactos_validos = df_contactos_normalizados[condicion_contacto_valido].copy()

df_contactos_revision = df_contactos_normalizados[~condicion_contacto_valido].copy()

In [98]:
print("Contactos válidos:", len(df_contactos_validos))
print("Contactos revisión:", len(df_contactos_revision))

Contactos válidos: 24
Contactos revisión: 5


### Paso 8 — Persistencia de Contactos en SQLite

Se persisten las tablas derivadas de Contactos en SQLite como almacenamiento intermedio reiniciable.

Tablas persistidas:

- contactos_normalizados
- contactos_validos
- contactos_revision

Objetivos:

- conservar resultados procesados
- evitar reprocesamiento innecesario
- preparar futura integración relacional
- permitir validación externa mediante SQL

La persistencia se realiza después de finalizar la normalización y validación de Contactos.

#### Paso 8.1 - Contactos normalizados a SQLite

In [99]:
df_contactos_normalizados.to_sql(
    "contactos_normalizados",
    conexion,
    if_exists="replace",
    index=False
)

29

#### Paso 8.2 - Contactos válidos a SQLite

In [100]:
df_contactos_validos.to_sql(
    "contactos_validos",
    conexion,
    if_exists="replace",
    index=False
)

24

#### Paso 8.3 - Contactos revisión a SQLite

In [101]:
df_contactos_revision.to_sql(
    "contactos_revision",
    conexion,
    if_exists="replace",
    index=False
)

5

### Paso 9 - Cálculo de métricas 

In [102]:
total_contactos = len(df_contactos_normalizados)
contactos_validos = len(df_contactos_validos)
contactos_revision = len(df_contactos_revision)

telefonos_incompletos = df_contactos_normalizados["telefono_incompleto_flag"].sum()
telefonos_anomalos = df_contactos_normalizados["telefono_anomalo_flag"].sum()

print("Total contactos:", total_contactos)
print("Contactos válidos:", contactos_validos)
print("Contactos revisión:", contactos_revision)
print("% válidos:", round(contactos_validos / total_contactos * 100, 2))
print("% revisión:", round(contactos_revision / total_contactos * 100, 2))
print("Teléfonos incompletos:", telefonos_incompletos)
print("Teléfonos anómalos:", telefonos_anomalos)

Total contactos: 29
Contactos válidos: 24
Contactos revisión: 5
% válidos: 82.76
% revisión: 17.24
Teléfonos incompletos: 2
Teléfonos anómalos: 5


In [103]:
reporte_contactos = pd.DataFrame([{
    "total_contactos": total_contactos,
    "contactos_validos": contactos_validos,
    "contactos_revision": contactos_revision,
    "porcentaje_validos": round(contactos_validos / total_contactos * 100, 2),
    "porcentaje_revision": round(contactos_revision / total_contactos * 100, 2),
    "telefonos_incompletos": telefonos_incompletos,
    "telefonos_anomalos": telefonos_anomalos
}])

reporte_contactos

,total_contactos,contactos_validos,contactos_revision,porcentaje_validos,porcentaje_revision,telefonos_incompletos,telefonos_anomalos
0,29,24,5,82.76,17.24,2,5


# Cierre ETAPA 5 — Contactos

## Objetivo cumplido

Se construyó y normalizó la entidad Contactos como tabla independiente del master de alumnos.

## Procesos realizados

- separación de Contactos desde `df_identidad_limpia`
- incorporación de `id_persona`
- conservación de `nombre_completo` como apoyo visual
- expansión de teléfonos múltiples
- normalización a granularidad: 1 fila = 1 teléfono
- detección de teléfonos incompletos
- separación de contactos válidos y contactos en revisión
- generación de `reporte_contactos`
- persistencia SQLite de tablas finales

## Outputs generados

- `df_contactos_normalizados`
- `df_contactos_validos`
- `df_contactos_revision`
- `reporte_contactos`

## Resultado

La entidad Contactos queda preparada para futura relación SQL mediante `id_persona`.

## Próximo bloque

Diseño de persistencia SQLite escalable y modularización del pipeline.

## Persistencia SQLite — Contactos y Master

Se persisten las tablas finales del bloque Alumnos + Contactos.

Durante el piloto se utiliza `if_exists="replace"` para garantizar reproducibilidad del notebook.

En etapa de escalamiento multi-curso, la estrategia deberá cambiar a persistencia incremental controlada, evitando duplicados y preservando `id_persona`.

In [104]:
df_identidad_limpia.to_sql(
    "identidad_limpia", 
    conexion, 
    if_exists="replace", 
    index=False
)

df_alumnos_master.to_sql(
    "alumnos_master",
    conexion,
    if_exists="replace",
    index=False
)

df_contactos_normalizados.to_sql(
    "contactos_normalizados",
    conexion,
    if_exists="replace",
    index=False
)

df_contactos_validos.to_sql(
    "contactos_validos",
    conexion,
    if_exists="replace",
    index=False
)

df_contactos_revision.to_sql(
    "contactos_revision",
    conexion,
    if_exists="replace",
    index=False
)

reporte_contactos.to_sql(
    "reporte_contactos",
    conexion,
    if_exists="replace",
    index=False
)

1

In [105]:
pd.read_sql("SELECT * FROM contactos_validos LIMIT 5", conexion)

,id_persona,nombre_completo,telefono_raw,telefono_num,tipo_contacto,telefono_multiple_flag,telefono_largo,telefono_anomalo_flag,telefono_incompleto_flag
0,55,"ACHIGAR, Tiziano",2932-520752M,2932520752,M,0,10,0,0
1,56,"AGUILAR, Ian Nicolás",2932-573441M,2932573441,M,0,10,0,0
2,57,"ALTAMIRANO, Joaquin Román",2932-614538M,2932614538,M,0,10,0,0
3,58,"ARANDA, Maia Mariel",2932-520048P,2932520048,P,0,10,0,0
4,59,"CARRERAS FLAMMINI, Paula Sofía",2932-505154M,2932505154,M,0,10,0,0
